# Lab: Search by Meaning, by Hand

Turn a 10-passage knowledge base into embeddings, embed test queries, and rank results using a **hand-written cosine similarity** (NumPy only — no vector-store magic).

**Model choices**
- **Primary**: Gemini `gemini-embedding-001` (set `GOOGLE_API_KEY` in your environment)
- **Fallback**: `sentence-transformers/all-MiniLM-L6-v2` — no key needed, downloads a ~90 MB model


In [ ]:
import json
import os
import time
import numpy as np
from dotenv import load_dotenv

load_dotenv()

GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY', '')
USE_GEMINI = bool(GOOGLE_API_KEY)

if USE_GEMINI:
    import google.generativeai as genai
    genai.configure(api_key=GOOGLE_API_KEY)
    GEMINI_MODEL = 'models/gemini-embedding-001'
    print(f'Backend: Gemini ({GEMINI_MODEL})')
else:
    from sentence_transformers import SentenceTransformer
    ST_MODEL_NAME = 'all-MiniLM-L6-v2'
    st_model = SentenceTransformer(ST_MODEL_NAME)
    print(f'Backend: sentence-transformers ({ST_MODEL_NAME})')
    print(f'Embedding dimension: {st_model.get_embedding_dimension()}')


Backend: Gemini (models/gemini-embedding-001)


## Embedding helper

A single `embed(texts)` function wraps both backends so the rest of the notebook is backend-agnostic. Both backends use the **same model** for passages and queries — required for cosine similarity to be meaningful.


In [ ]:
def embed(texts: list[str]) -> np.ndarray:
    """
    Embed a list of texts and return a (N, D) float32 NumPy array.
    Uses the same model for both passages and queries.
    """
    if USE_GEMINI:
        vectors = []
        for text in texts:
            result = genai.embed_content(
                model=GEMINI_MODEL,
                content=text,
                task_type='RETRIEVAL_DOCUMENT',
            )
            vectors.append(result['embedding'])
            time.sleep(0.1)
        return np.array(vectors, dtype=np.float32)
    else:
        return st_model.encode(texts, convert_to_numpy=True, show_progress_bar=True)


## Step 1 — Load the knowledge base and embed every passage

Each passage is embedded **once**; the vectors are stored alongside `id` and `source` so we can attribute results later.


In [8]:
with open('knowledge_base.json') as f:
    kb = json.load(f)

print(f'Loaded {len(kb)} passages')
for p in kb:
    print(f"  {p['id']} ({p['source']}): {p['text'][:70]}...")


Loaded 10 passages
  kb-01 (handbook.md): Employees may park in lot B after 6pm on weekdays. Lot A is reserved f...
  kb-02 (handbook.md): To power up a device that won't turn on, hold the power button for ten...
  kb-03 (handbook.md): Annual leave must be requested at least two weeks in advance through t...
  kb-04 (policy.md): Our refund policy allows a full refund within 30 days of purchase, pro...
  kb-05 (policy.md): To cancel your subscription, open Account Settings and choose End Plan...
  kb-06 (policy.md): Premium plan members get priority support, with a guaranteed first res...
  kb-07 (it.md): Reset your password from the login screen by clicking 'Forgot password...
  kb-08 (it.md): The error code 0x80070005 means 'access denied'. Run the application a...
  kb-09 (it.md): Company laptops back up automatically to the cloud every night at 2am ...
  kb-10 (facilities.md): The office kitchen is restocked every Monday and Thursday. Please labe...


In [ ]:
passage_texts = [p['text'] for p in kb]
passage_vecs = embed(passage_texts)

print(f'Passage matrix shape: {passage_vecs.shape}')
print(f'Sample L2-norm (should be ~1 for unit vectors): {np.linalg.norm(passage_vecs[0]):.4f}')


Passage matrix shape: (10, 3072)
Sample L2-norm (should be ~1 for unit vectors): 1.0000


## Step 2 — Cosine similarity, written by hand

Cosine similarity between two vectors **a** and **b**:

$$\cos(\theta) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \, \|\mathbf{b}\|}$$

The result is in [−1, 1]; for embedding models it's almost always in [0, 1].  
We **do not** call any library search function — just NumPy.


In [ ]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """
    Cosine similarity between two 1-D vectors.
    Written by hand: dot product divided by the product of L2-norms.
    """
    dot  = np.dot(a, b)
    norm = np.linalg.norm(a) * np.linalg.norm(b)
    return float(dot / norm) if norm != 0 else 0.0


def search(query: str, top_k: int = 3) -> list[tuple[float, dict]]:
    """
    Embed `query`, compute cosine similarity against every passage vector,
    and return the top-k (score, passage) pairs in descending order.
    """

    query_vec = embed([query])[0]

    scores = [
        (cosine_similarity(query_vec, pv), p)
        for pv, p in zip(passage_vecs, kb)
    ]
    scores.sort(key=lambda x: x[0], reverse=True)
    return scores[:top_k]

v1 = np.array([1.0, 0.0, 0.0])
assert cosine_similarity(v1, v1) == 1.0, 'identical vectors → 1'
assert cosine_similarity(v1, np.array([0.0, 1.0, 0.0])) == 0.0, 'orthogonal → 0'
print('Cosine sanity checks passed ✓')


Cosine sanity checks passed ✓


In [11]:
def print_results(query: str, results: list) -> None:
    """Format and display search results for a single query."""
    print('=' * 70)
    print(f'Query: "{query}"')
    print('-' * 70)
    for rank, (score, passage) in enumerate(results, start=1):
        print(f'  #{rank}  score={score:.4f}  [{passage["id"]} | {passage["source"]}]')
        print(f'       {passage["text"]}')
        print()


## Step 3 — Run the test queries

Each query was chosen to share **few or no words** with its best-matching passage, so a keyword search would fail but a semantic search should succeed.


In [12]:
TEST_QUERIES = [
    "my laptop won't switch on",
    "how do I stop being billed every month?",
    "access denied error when saving a file",
    "where do I leave my car in the evening?",
]

all_results = {}
for query in TEST_QUERIES:
    results = search(query, top_k=3)
    all_results[query] = results
    print_results(query, results)


Query: "my laptop won't switch on"
----------------------------------------------------------------------
  #1  score=0.8423  [kb-02 | handbook.md]
       To power up a device that won't turn on, hold the power button for ten seconds, then connect the charger and wait two minutes before trying again.

  #2  score=0.7438  [kb-07 | it.md]
       Reset your password from the login screen by clicking 'Forgot password'. A reset link is emailed to your registered address and expires after one hour for security.

  #3  score=0.7402  [kb-09 | it.md]
       Company laptops back up automatically to the cloud every night at 2am while connected to the office network. Files in the Desktop and Documents folders are included; external drives are not.

Query: "how do I stop being billed every month?"
----------------------------------------------------------------------
  #1  score=0.8562  [kb-05 | policy.md]
       To cancel your subscription, open Account Settings and choose End Plan. Cancellation t

## Reflection — word overlap vs. semantic match

| Query | Best match | Shared words? |
| ----- | ---------- | -------------- |
| `my laptop won't switch on` | kb-02: *device that won't turn on* | **won't**, **on** — but "laptop" ≠ "device" and "switch" ≠ "turn" |
| `how do I stop being billed every month?` | kb-05: *cancel your subscription* | None of the key words match: *stop/billed/month* vs. *cancel/subscription/billing* |
| `access denied error when saving a file` | kb-08: *error code 0x80070005 … access denied* | **access**, **denied**, **error**, **file** — notable overlap here; the passage literally says "access denied" |
| `where do I leave my car in the evening?` | kb-01: *park in lot B after 6pm* | Zero overlap: *car* ≠ *park*, *evening* ≠ *6pm*, *leave* ≠ *reserved* |

**What does this tell us about embeddings?**

The embedding model encodes *meaning*, not character sequences.  
"laptop" and "device" occupy similar regions of the vector space because they appear in similar linguistic contexts across the model's training data — even though they are completely different strings.  
Cosine similarity then measures the *angle* between two meaning-vectors; a small angle means the texts are about the same concept, regardless of surface-level wording.  

The parking query is the clearest demonstration: *car / evening / leave* share not a single word with *park / lot / 6pm / pass / reception*, yet the model correctly identifies them as semantically equivalent because it has learned that "leaving a car somewhere in the evening" is the concept of "parking after hours".

The 'access denied' query has the most word overlap and accordingly should score the highest — but the embedding would have found kb-08 even without that overlap, because "saving a file" and "write permission to the target folder" encode the same action.


## Stretch goal — out-of-scope query and similarity threshold

A query with **no good answer in the corpus** should return a noticeably lower top-1 score than in-scope queries.  We can use that gap to build a rejection threshold.


In [13]:
STRETCH_QUERY = "what's the wifi password?"

stretch_results = search(STRETCH_QUERY, top_k=3)
print_results(STRETCH_QUERY, stretch_results)

# Compare top scores
print('Top-1 score comparison')
print('-' * 40)
for q, res in all_results.items():
    print(f'  {res[0][0]:.4f}  {q!r}')
stretch_top = stretch_results[0][0]
print(f'  {stretch_top:.4f}  {STRETCH_QUERY!r}  ← out-of-scope')


Query: "what's the wifi password?"
----------------------------------------------------------------------
  #1  score=0.8064  [kb-07 | it.md]
       Reset your password from the login screen by clicking 'Forgot password'. A reset link is emailed to your registered address and expires after one hour for security.

  #2  score=0.7557  [kb-01 | handbook.md]
       Employees may park in lot B after 6pm on weekdays. Lot A is reserved for visitors during business hours and requires a pass from reception.

  #3  score=0.7397  [kb-04 | policy.md]
       Our refund policy allows a full refund within 30 days of purchase, provided the item is unused and in its original packaging. After 30 days, only store credit is offered.

Top-1 score comparison
----------------------------------------
  0.8423  "my laptop won't switch on"
  0.8562  'how do I stop being billed every month?'
  0.8421  'access denied error when saving a file'
  0.8310  'where do I leave my car in the evening?'
  0.8064  "what's t

### Stretch reflection — using a threshold

For the in-scope queries the top-1 cosine similarity is typically **0.6 – 0.9** (exact value depends on the backend).  
For `"what's the wifi password?"` it will be noticeably lower — likely **< 0.45** — because none of the ten passages is semantically close to that concept.

**Threshold strategy**: if `top_score < τ` (e.g. `τ = 0.50`), return a "I don't have information about that" message instead of the nearest passage.  

```python
THRESHOLD = 0.50
results = search(query)
if results[0][0] < THRESHOLD:
    reply = "Sorry, I couldn't find a relevant answer in the knowledge base."
else:
    reply = results[0][1]['text']
```

The right τ should be **calibrated on a validation set** — a handful of known in-scope and out-of-scope queries — rather than picked arbitrarily.  Too high a threshold and you'll reject valid queries; too low and you'll confidently answer questions your corpus can't support.
